# Summary Report Demo

This notebook demonstrates using `nvcl_kit.plots.create_summary_report()` to produce
a multi-page PDF spectral summary report for a single borehole.

In [ ]:
from nvcl_kit.param_builder import param_builder
from nvcl_kit.reader import NVCLReader
from nvcl_kit.generators import gen_summary_dataframe
from nvcl_kit.plots import create_summary_report

## 1. Connect to the NSW NVCL node

In [ ]:
param = param_builder("NSW")
if not param:
    raise RuntimeError("Failed to setup connection parameters")

reader = NVCLReader(param)
if not reader.wfs:
    raise RuntimeError("Cannot contact NSW NVCL service")

print("Connected to NSW NVCL node.")

## 2. Retrieve borehole metadata and spectral data

In [ ]:
boreholeid = "MIN_331641"
swir_scalar_set = "uTSAS"
tir_scalar_set = "ujCLST"
resolution = 1.0

# Retrieve borehole feature metadata
bh_meta_list = reader.filter_feat_list(nvcl_id=boreholeid)
bh_meta = bh_meta_list[0]

print(f"Borehole: {bh_meta.name} ({boreholeid})")
print(f"  Depth: {bh_meta.boreholeLength_m} m")
print(f"  Location: ({bh_meta.x}, {bh_meta.y})")

In [ ]:
# Download SWIR summary data
swir_result = next(
    gen_summary_dataframe(
        reader=reader,
        nvcl_id_list=[boreholeid],
        scalar_set=swir_scalar_set,
        scalar_level="group",
        start_depth="floor",
        weighted=True,
        resolution=resolution,
        include_srss=True,
        include_snr=True,
        continue_on_missing=True,
    ),
    None,
)

if swir_result is None:
    raise RuntimeError(f"Failed to retrieve SWIR data for {boreholeid}")

swir_meta, swir_df = swir_result
print(f"SWIR data: {len(swir_df)} rows, columns: {list(swir_df.columns)}")

In [ ]:
# Download TIR summary data
tir_result = next(
    gen_summary_dataframe(
        reader=reader,
        nvcl_id_list=[boreholeid],
        scalar_set=tir_scalar_set,
        scalar_level="group",
        start_depth="floor",
        weighted=True,
        resolution=resolution,
        include_srss=True,
        include_snr=True,
        continue_on_missing=True,
    ),
    None,
)

if tir_result is None:
    raise RuntimeError(f"Failed to retrieve TIR data for {boreholeid}")

tir_meta, tir_df = tir_result
print(f"TIR data: {len(tir_df)} rows, columns: {list(tir_df.columns)}")

## 3. Build colour map and classification column lists

In [ ]:
# Merge SWIR and TIR classification colours into one dict
colours = {name: info["colour"] for name, info in swir_meta["classifications"].items()}
colours.update({name: info["colour"] for name, info in tir_meta["classifications"].items()})

# Determine which classification columns are actually present in each DataFrame
swir_cols = [col for col in swir_meta["classifications"].keys() if col in swir_df.columns]
tir_cols = [col for col in tir_meta["classifications"].keys() if col in tir_df.columns]

print(f"SWIR groups: {swir_cols}")
print(f"TIR groups: {tir_cols}")

## 4. Generate the PDF report

In [ ]:
output_file = f"{boreholeid}_summary_report.pdf"

figs = create_summary_report(
    boreholeid,
    swir_df,
    tir_df,
    swir_cols,
    tir_cols,
    colours=colours,
    hole_overview=True,
    swir_scalar_set=swir_scalar_set,
    tir_scalar_set=tir_scalar_set,
    output_path=output_file,
    name=bh_meta.name if bh_meta.name != "" else None,
    year_drilled=bh_meta.drillStartDate if bh_meta.drillStartDate != "" else None,
    drill_type=bh_meta.drillingMethod if bh_meta.drillingMethod != "" else None,
    total_depth=bh_meta.boreholeLength_m if bh_meta.boreholeLength_m != "" else None,
    longitude=bh_meta.x if bh_meta.x != "" else None,
    latitude=bh_meta.y if bh_meta.y != "" else None,
    crs="GDA94",
)

print(f"\nReport written to: {output_file}")
print(f"Total pages: {len(figs)}")